# Phase 0: Mathematical & Computational Foundations  
## Notebook 00_02 — Brownian Motion Simulation, Scaling Limits, and Quadratic Variation

### Research question

How does a discrete random walk become Brownian motion in the continuous-time limit, and why is Brownian motion the central noise process in mathematical finance?

This notebook builds the bridge from the previous martingale notebook to continuous-time stochastic calculus. It focuses on:

1. **Scaled random walks** and the intuition behind Donsker's theorem.
2. **Brownian motion simulation** using normally distributed increments.
3. **Independent increments** and terminal normality.
4. **Quadratic variation**, the key path property behind Itô calculus.
5. **Geometric Brownian motion**, the basic continuous-time asset price model used in Black-Scholes-Merton.

The aim is to make Brownian motion computationally visible before using it in later notebooks on stochastic differential equations, option pricing, Monte Carlo methods, and volatility modelling.

## 1. Theory: what is Brownian motion?

A standard Brownian motion $(W_t)_{t \geq 0}$ is a continuous-time stochastic process satisfying:

1. $W_0 = 0$;
2. $W_t - W_s \sim \mathcal N(0, t-s)$ for $0 \leq s < t$;
3. increments over non-overlapping time intervals are independent;
4. sample paths are continuous but almost surely nowhere differentiable.

The key simulation rule is:

$$
W_{t+\Delta t} - W_t \sim \mathcal N(0, \Delta t)
$$

Equivalently:

$$
\Delta W_t = \sqrt{\Delta t} Z
$$

where:

$$
Z \sim \mathcal N(0,1)
$$

This square-root scaling is central. Brownian increments scale like $\sqrt{\Delta t}$, not like $\Delta t$. That is why Brownian motion has irregular paths and non-zero quadratic variation.

## 2. From random walk to Brownian motion

Start with a simple symmetric random walk:

$$
S_n = \sum_{i=1}^{n} \xi_i
$$

where:

$$
\mathbb P(\xi_i = 1) = \mathbb P(\xi_i = -1) = \frac{1}{2}
$$

To obtain a continuous-time approximation, rescale time and space:

$$
W_t^{(n)} = \frac{S_{\lfloor nt \rfloor}}{\sqrt{n}}
$$

As $n$ becomes large, this scaled random walk converges in distribution to Brownian motion.

This result is known as **Donsker's theorem**, or the functional central limit theorem.

The intuition is:

- time is sped up by $n$;
- space is scaled down by $\sqrt n$;
- the limiting process has Gaussian increments and continuous paths.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt

rng = np.random.default_rng(seed=42)

In [ ]:
def simulate_scaled_random_walk(num_steps: int, terminal_time: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """
    Simulate one scaled symmetric random walk approximation to Brownian motion.

    W_t^(n) = S_floor(nt) / sqrt(n)

    Parameters
    ----------
    num_steps:
        Number of discrete random-walk steps.

    terminal_time:
        Final time horizon.

    Returns
    -------
    time_grid:
        Array of time points.

    scaled_path:
        Scaled random-walk path.
    """
    dt = terminal_time / num_steps
    steps = rng.choice([-1, 1], size=num_steps)
    raw_walk = np.insert(np.cumsum(steps), 0, 0)

    time_grid = np.linspace(0, terminal_time, num_steps + 1)
    scaled_path = np.sqrt(dt) * raw_walk

    return time_grid, scaled_path

In [ ]:
plt.figure(figsize=(10, 6))

for num_steps in [50, 200, 1000, 5000]:
    time_grid, scaled_path = simulate_scaled_random_walk(num_steps=num_steps, terminal_time=1.0)
    plt.plot(time_grid, scaled_path, label=f"n = {num_steps}", alpha=0.8)

plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Scaled Random Walks Approaching Brownian Motion")
plt.xlabel("Time")
plt.ylabel(r"$W_t^{(n)}$")
plt.legend()
plt.show()

## 3. Direct Brownian motion simulation

To simulate Brownian motion directly on a time grid:

$$
0 = t_0 < t_1 < \cdots < t_N = T
$$

we generate increments:

$$
\Delta W_i = W_{t_i} - W_{t_{i-1}} = \sqrt{\Delta t}Z_i
$$

where:

$$
Z_i \sim \mathcal N(0,1)
$$

Then:

$$
W_{t_i} = \sum_{j=1}^{i} \Delta W_j
$$

In [ ]:
def simulate_brownian_motion(
    num_paths: int,
    num_steps: int,
    terminal_time: float = 1.0
) -> tuple[np.ndarray, np.ndarray]:
    """
    Simulate standard Brownian motion paths.

    Parameters
    ----------
    num_paths:
        Number of independent Brownian paths.

    num_steps:
        Number of time steps.

    terminal_time:
        Final time horizon T.

    Returns
    -------
    time_grid:
        Array of shape (num_steps + 1,).

    paths:
        Array of shape (num_paths, num_steps + 1).
    """
    dt = terminal_time / num_steps
    time_grid = np.linspace(0, terminal_time, num_steps + 1)

    increments = np.sqrt(dt) * rng.standard_normal(size=(num_paths, num_steps))

    paths = np.zeros((num_paths, num_steps + 1))
    paths[:, 1:] = np.cumsum(increments, axis=1)

    return time_grid, paths

In [ ]:
num_paths = 20_000
num_steps = 1_000
terminal_time = 1.0

time_grid, brownian_paths = simulate_brownian_motion(
    num_paths=num_paths,
    num_steps=num_steps,
    terminal_time=terminal_time
)

brownian_paths.shape

In [ ]:
plt.figure(figsize=(10, 6))

for i in range(50):
    plt.plot(time_grid, brownian_paths[i], alpha=0.35)

plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Simulated Brownian Motion Paths")
plt.xlabel("Time")
plt.ylabel(r"$W_t$")
plt.show()

## 4. Checking terminal normality

For standard Brownian motion:

$$
W_T \sim \mathcal N(0,T)
$$

So at $T=1$, we should have approximately:

$$
W_1 \sim \mathcal N(0,1)
$$

The sample mean should be close to 0 and the sample variance should be close to 1.

In [ ]:
terminal_values = brownian_paths[:, -1]

summary_stats = pd.Series({
    "sample_mean_W_T": terminal_values.mean(),
    "sample_variance_W_T": terminal_values.var(ddof=1),
    "theoretical_mean": 0.0,
    "theoretical_variance": terminal_time
})

summary_stats

In [ ]:
x = np.linspace(-4, 4, 500)
normal_density = (1 / np.sqrt(2 * np.pi * terminal_time)) * np.exp(-(x ** 2) / (2 * terminal_time))

plt.figure(figsize=(10, 6))
plt.hist(terminal_values, bins=70, density=True, alpha=0.7, label="Simulated terminal values")
plt.plot(x, normal_density, linewidth=2, label=r"Theoretical $\mathcal{N}(0,T)$ density")
plt.title(r"Terminal Distribution of $W_T$")
plt.xlabel(r"$W_T$")
plt.ylabel("Density")
plt.legend()
plt.show()

## 5. Checking independent increments

Brownian motion has independent increments. This means that changes over non-overlapping time intervals should be approximately uncorrelated in simulation.

For example:

$$
W_{0.4} - W_{0.2}
$$

and

$$
W_{0.8} - W_{0.6}
$$

should have correlation close to zero across many simulated paths.

In [ ]:
def get_time_index(time_grid: np.ndarray, t: float) -> int:
    """
    Return the index in the time grid closest to time t.
    """
    return int(np.argmin(np.abs(time_grid - t)))


i_02 = get_time_index(time_grid, 0.2)
i_04 = get_time_index(time_grid, 0.4)
i_06 = get_time_index(time_grid, 0.6)
i_08 = get_time_index(time_grid, 0.8)

increment_1 = brownian_paths[:, i_04] - brownian_paths[:, i_02]
increment_2 = brownian_paths[:, i_08] - brownian_paths[:, i_06]

increment_correlation = np.corrcoef(increment_1, increment_2)[0, 1]
increment_correlation

The estimated correlation should be close to zero.

This is not a proof of independence, but it is a useful computational sanity check.

## 6. Quadratic variation

Brownian motion has a strange and important path property.

Its **quadratic variation** over $[0,T]$ is:

$$
[W]_T = \lim_{\|\Pi\| \to 0} \sum_{i=1}^{n} (W_{t_i} - W_{t_{i-1}})^2 = T
$$

This is one of the key reasons that Itô calculus differs from ordinary calculus.

For a smooth deterministic function, squared increments become negligible as the partition gets finer.

For Brownian motion, squared increments accumulate to a non-zero limit.

This is the intuition behind the famous symbolic rule:

$$
(dW_t)^2 = dt
$$

In [ ]:
def quadratic_variation(path: np.ndarray) -> float:
    """
    Compute realised quadratic variation along a discrete path.
    """
    increments = np.diff(path)
    return np.sum(increments ** 2)


def total_variation(path: np.ndarray) -> float:
    """
    Compute realised total variation along a discrete path.
    """
    increments = np.diff(path)
    return np.sum(np.abs(increments))

In [ ]:
# Simulate one fine Brownian path and compare realised variation under increasingly fine partitions.
fine_steps = 20_000
fine_time_grid, fine_paths = simulate_brownian_motion(
    num_paths=1,
    num_steps=fine_steps,
    terminal_time=1.0
)

fine_path = fine_paths[0]

partition_sizes = [50, 100, 250, 500, 1000, 2500, 5000, 10000, 20000]

variation_rows = []

for n in partition_sizes:
    stride = fine_steps // n
    sampled_path = fine_path[::stride]

    # Ensure the terminal point is included.
    if len(sampled_path) > n + 1:
        sampled_path = sampled_path[:n + 1]

    variation_rows.append({
        "num_intervals": len(sampled_path) - 1,
        "quadratic_variation": quadratic_variation(sampled_path),
        "total_variation": total_variation(sampled_path)
    })

variation_df = pd.DataFrame(variation_rows)
variation_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(variation_df["num_intervals"], variation_df["quadratic_variation"], marker="o", label="Realised quadratic variation")
plt.axhline(terminal_time, linestyle="--", label="Theoretical limit T")
plt.xscale("log")
plt.title("Quadratic Variation of a Brownian Path")
plt.xlabel("Number of partition intervals")
plt.ylabel("Quadratic variation")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(variation_df["num_intervals"], variation_df["total_variation"], marker="o", label="Realised total variation")
plt.xscale("log")
plt.title("Total Variation of a Brownian Path")
plt.xlabel("Number of partition intervals")
plt.ylabel("Total variation")
plt.legend()
plt.show()

## 7. Why this matters: Itô intuition

For a smooth deterministic path, if $\Delta t$ is small, then:

$$
(\Delta x)^2
$$

is much smaller than $\Delta t$, so second-order terms usually disappear in ordinary calculus.

For Brownian motion:

$$
\Delta W_t \sim \sqrt{\Delta t}
$$

so:

$$
(\Delta W_t)^2 \sim \Delta t
$$

This means second-order terms survive.

That is the core intuition behind Itô's formula. For a function $f(t, W_t)$, the extra second-derivative term appears because Brownian motion has non-zero quadratic variation.

This is why stochastic calculus is not just ordinary calculus with random variables added.

## 8. Bridge to finance: geometric Brownian motion

A basic continuous-time asset price model is geometric Brownian motion:

$$
dS_t = \mu S_t dt + \sigma S_t dW_t
$$

Its exact solution is:

$$
S_t = S_0 \exp\left( \left(\mu - \frac{1}{2}\sigma^2\right)t + \sigma W_t \right)
$$

This model keeps prices positive and is the underlying process behind the Black-Scholes-Merton framework.

The term:

$$
-\frac{1}{2}\sigma^2
$$

appears because of Itô calculus and the quadratic variation of Brownian motion.

In [ ]:
def simulate_geometric_brownian_motion(
    num_paths: int,
    num_steps: int,
    terminal_time: float,
    s0: float,
    mu: float,
    sigma: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Simulate geometric Brownian motion using the exact solution.

    dS_t = mu S_t dt + sigma S_t dW_t
    """
    time_grid, brownian_paths = simulate_brownian_motion(
        num_paths=num_paths,
        num_steps=num_steps,
        terminal_time=terminal_time
    )

    drift = (mu - 0.5 * sigma ** 2) * time_grid
    diffusion = sigma * brownian_paths

    price_paths = s0 * np.exp(drift + diffusion)

    return time_grid, price_paths

In [ ]:
gbm_time, gbm_paths = simulate_geometric_brownian_motion(
    num_paths=2_000,
    num_steps=1_000,
    terminal_time=1.0,
    s0=100.0,
    mu=0.08,
    sigma=0.20
)

plt.figure(figsize=(10, 6))

for i in range(50):
    plt.plot(gbm_time, gbm_paths[i], alpha=0.35)

plt.title("Geometric Brownian Motion Sample Paths")
plt.xlabel("Time")
plt.ylabel(r"$S_t$")
plt.show()

## 9. Limitations

This notebook deliberately uses idealised Brownian motion.

### 9.1 Continuous paths exclude jumps

Standard Brownian motion has continuous paths. Real financial markets can experience sudden jumps caused by news, liquidity shocks, macroeconomic releases, or limit moves.

Later notebooks can introduce jump-diffusion models.

### 9.2 Gaussian increments underestimate extreme events

Brownian motion has normally distributed increments. Empirical financial returns often have heavier tails than the normal distribution.

This motivates Student-t models, jump processes, stochastic volatility, and stress testing.

### 9.3 Constant volatility is unrealistic

Geometric Brownian motion assumes constant volatility. Real markets display volatility clustering, volatility smiles, and term structures.

This motivates GARCH, Heston, local volatility, and rough volatility models.

### 9.4 Brownian motion is Markovian

Brownian motion has no memory: conditional on the present, the past does not affect the future distribution.

Many modern volatility models introduce path dependence or memory, especially through rough volatility and Volterra-type processes.

### 9.5 Simulation depends on discretisation

Although Brownian motion can be simulated exactly on a fixed grid, SDEs driven by Brownian motion often require numerical schemes such as Euler-Maruyama or Milstein.

Discretisation error becomes important in Monte Carlo pricing and hedging.

## 10. Research frontier and current directions

Brownian motion is classical, but it remains the starting point for many modern research directions in mathematical and computational finance.

### 10.1 Rough volatility and fractional Brownian motion

Classical Brownian-driven models are Markovian and often too smooth in their volatility dynamics.

Rough volatility models use processes related to fractional Brownian motion or Volterra kernels to capture the rough, persistent behaviour observed in volatility data.

A possible future notebook could compare standard Brownian motion with fractional Brownian motion for different Hurst exponents, then connect this to the rough Bergomi model.

### 10.2 Neural SDEs and arbitrage-free machine learning

Neural SDE models use neural networks to parameterise drift and diffusion functions.

The challenge in finance is not only to fit data, but to preserve structural constraints such as no-arbitrage, positivity, and realistic option surface dynamics.

A possible future notebook could compare an unconstrained neural SDE with an arbitrage-aware model on synthetic option data.

### 10.3 Multilevel Monte Carlo

Brownian simulation is a core ingredient in Monte Carlo derivative pricing.

However, ordinary Monte Carlo can be computationally expensive. Multilevel Monte Carlo reduces cost by combining many cheap coarse simulations with fewer expensive fine simulations.

A possible future notebook could price an Asian option using both standard Monte Carlo and multilevel Monte Carlo, comparing error versus runtime.

### 10.4 Rough paths and non-Markovian pricing

Some modern models are path-dependent or non-Markovian, making classical PDE methods harder to apply.

Rough path theory provides tools for analysing differential equations driven by irregular signals, including Brownian-like paths and rough volatility models.

A possible future notebook could introduce signatures of paths as features for financial time series.

### 10.5 Quantum and accelerated Monte Carlo

Monte Carlo simulation is widely used in risk management and derivatives pricing, but its convergence rate is slow.

Research into quantum amplitude estimation and accelerated Monte Carlo methods investigates whether high-dimensional pricing and risk problems can be computed faster in future hardware regimes.

A possible future notebook could benchmark standard Monte Carlo, quasi-Monte Carlo, and variance reduction techniques before discussing quantum methods conceptually.

## 11. Suggested follow-up notebooks

This notebook naturally leads to:

1. **00_03_ito_calculus_numerical_intuition**  
   Itô sums, stochastic integrals, and why evaluation point matters.

2. **02_03_black_scholes_merton_pde**  
   Deriving the Black-Scholes-Merton PDE from a Brownian-driven asset model.

3. **02_06_monte_carlo_gbm_and_fat_tails**  
   Monte Carlo simulation under GBM and heavier-tailed alternatives.

4. **02_10_finite_difference_pde_pricer**  
   Solving pricing PDEs numerically.

5. **02_11_heston_model_calibration**  
   Moving from constant volatility to stochastic volatility.

6. **02_13_multilevel_monte_carlo_pricing**  
   Reducing simulation cost for derivative pricing.

7. **06_05_rough_volatility_estimation**  
   Estimating roughness and memory in volatility time series.

## 12. Summary

This notebook simulated Brownian motion and verified several of its defining properties:

1. Brownian increments scale like $\sqrt{\Delta t}$.
2. Terminal values satisfy:

$$
W_T \sim \mathcal N(0,T)
$$

3. Non-overlapping increments are approximately independent.
4. Quadratic variation converges to:

$$
[W]_T = T
$$

5. Total variation grows as the partition becomes finer.
6. Geometric Brownian motion uses Brownian motion to create a positive asset price model.

The key mathematical takeaway is:

> Brownian motion is continuous but extremely irregular. Its non-zero quadratic variation is the reason stochastic calculus differs from ordinary calculus.

The key financial takeaway is:

> Brownian motion is the classical noise source behind Black-Scholes-Merton, Monte Carlo pricing, and many continuous-time finance models, but modern research often extends it to handle jumps, stochastic volatility, rough volatility, path dependence, and computational constraints.

## 13. Further reading

This section records the main directions connected to this foundational notebook.

### Textbook foundations

- Shreve, S. E. *Stochastic Calculus for Finance II: Continuous-Time Models.*
- Øksendal, B. *Stochastic Differential Equations: An Introduction with Applications.*
- Glasserman, P. *Monte Carlo Methods in Financial Engineering.*

### Research directions

- Gatheral, J., Jaisson, T., and Rosenbaum, M. "Volatility is Rough."
- Bayer, C., Friz, P., and Gatheral, J. "Pricing under rough volatility."
- Cohen, S. N., Reisinger, C., and Wang, S. "Arbitrage-Free Neural-SDE Market Models."
- Higham, D. J. and Giles, M. B. "Multilevel Monte Carlo methods for applications in finance."
- Buehler, H., Gonon, L., Teichmann, J., and Wood, B. "Deep Hedging."